# **Data Subsetting — Flight Delay Dataset (2022)**
*One-time prep: load the full 2022 CSV, subset to 1M rows, verify date spread, save as a working file.*

In [ ]:
import polars as pl

df = pl.read_csv("../data/flightsData.csv", low_memory=False)
print(df.shape)


(4078318, 61)


In [9]:
# subset to first 1M rows (or random sample if biased — check below first)
df_subset = df.sample(n=1_000_000, seed=42)
print(df_subset.shape)

(1000000, 61)


In [10]:
# verify date spread across months
df_subset = df_subset.with_columns(
    pl.col("FlightDate").str.to_date().alias("FlightDate")
)
print(df_subset["FlightDate"].min(), "to", df_subset["FlightDate"].max())


2022-01-01 to 2022-07-31


In [11]:
month_counts = (
    df_subset
    .with_columns(pl.col("FlightDate").dt.month().alias("month"))
    .group_by("month")
    .agg(pl.len().alias("count"))
    .sort("month")
)
print(month_counts)

shape: (7, 2)
┌───────┬────────┐
│ month ┆ count  │
│ ---   ┆ ---    │
│ i8    ┆ u32    │
╞═══════╪════════╡
│ 1     ┆ 138584 │
│ 2     ┆ 127414 │
│ 3     ┆ 144803 │
│ 4     ┆ 142015 │
│ 5     ┆ 147684 │
│ 6     ┆ 147562 │
│ 7     ┆ 151938 │
└───────┴────────┘


In [13]:
# save
df_subset.write_csv("../data/flightsData1m.csv")
print("Saved.")

Saved.
